INSTALAÇÃO DAS BIBLIOTECAS

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

FASE 1: INTERPRETAÇÃO AUTOMÁTICA DO DIAGRAMA DE ARQUITETURA (ADAPTA ONE VISION)

FASE 2: GERAÇÃO DO RELATÓRIO STRIDE (API CHATGPT)

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Inicializa o cliente OpenAI com chave de API
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_stride_report(architecture_description: str, architecture_name: str) -> str:
    """
    Gera um relatório de modelagem de ameaças STRIDE para uma arquitetura,
    utilizando a API do ChatGPT.

    Args:
        architecture_description (str): Uma descrição detalhada da arquitetura.
        architecture_name (str): Nome amigável da arquitetura para o relatório.

    Returns:
        str: O relatório de modelagem de ameaças STRIDE completo em formato Markdown.
    """
    prompt = f"""
    Você é um especialista em segurança cibernética e modelagem de ameaças, com foco
    em sistemas financeiros. Sua tarefa é analisar a seguinte arquitetura de sistema
    e gerar um relatório completo de modelagem de ameaças utilizando a metodologia STRIDE.

    A descrição da arquitetura foi gerada por uma ferramenta de interpretação de diagramas,
    identificando explicitamente os componentes como:
    - Usuários (tipos ou papéis)
    - Servidores (instâncias, microsserviços, funções serverless)
    - Bancos de dados (tipos, dados armazenados)
    - Serviços de rede (load balancers, firewalls, gateways)
    - Armazenamento (object storage, volumes)
    - Filas de mensagens
    - Integrações externas
    - Serviços de segurança (WAF, Shield, IAM, Entra ID)
    - Gerenciamento de APIs (API Gateway, Developer Portal)
    - Orquestração (Logic Apps, Lambda)

    Para cada uma das seis categorias STRIDE (Spoofing, Tampering, Repudiation,
    Information Disclosure, Denial of Service, Elevation of Privilege), faça o seguinte:
    1.  **Identifique ameaças potenciais** específicas para os componentes e fluxos descritos na arquitetura.
    2.  **Descreva o impacto** potencial de cada ameaça no contexto de um sistema financeiro.
    3.  **Sugira mitigações** concretas e práticas, alinhadas às melhores práticas de segurança para sistemas em nuvem e financeiros.

    Organize o relatório de forma clara e profissional, utilizando títulos e subtítulos em formato Markdown (## para seções, ### para subseções, **para negrito, * para listas).
    Seja o mais específico possível sobre os componentes envolvidos nas ameaças e mitigações.

    ---
    **Nome da Arquitetura:** {architecture_name}

    **Descrição da Arquitetura (proveniente da interpretação da imagem):**
    {architecture_description}
    ---
    """

    print(f"\nEnviando descrição da arquitetura '{architecture_name}' para a API da OpenAI para análise STRIDE...")

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "Você é um analista de segurança com expertise em modelagem de ameaças STRIDE para sistemas financeiros. Seja completo, objetivo e forneça exemplos práticos de mitigação. Não invente detalhes que não estão na descrição da arquitetura, mas use seu conhecimento para inferir ameaças e mitigacoes plausíveis. O resultado deve ser formatado em Markdown."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.6,
            max_tokens=4000 # Aumentado para garantir relatórios mais completos no docx
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro ao chamar a API da OpenAI: {e}"

def append_markdown_to_docx_document(document: Document, markdown_content: str, section_title: str):
    """
    Adiciona o conteúdo Markdown de um relatório a um documento .docx existente.
    Converte cabeçalhos (##, ###), listas e negrito simples.

    Args:
        document (Document): O objeto Document ao qual o conteúdo será adicionado.
        markdown_content (str): O conteúdo do relatório em formato Markdown.
        section_title (str): O título desta seção de arquitetura dentro do documento.
    """
    # Adiciona um título de seção para a arquitetura atual
    document.add_heading(section_title, level=1)
    document.add_paragraph()

    lines = markdown_content.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if line.startswith('### '):
            document.add_heading(line[4:], level=3)
        elif line.startswith('## '):
            document.add_heading(line[3:], level=2)
        elif line.startswith('* ') or line.startswith('- '):
            p = document.add_paragraph(style='List Bullet')
            
            parts = line[2:].split('**')
            if len(parts) > 1:
                for i, part in enumerate(parts):
                    run = p.add_run(part)
                    if i % 2 != 0:
                        run.bold = True
            else:
                p.add_run(line[2:])
        else:
            
            p = document.add_paragraph()
            parts = line.split('**')
            if len(parts) > 1:
                for i, part in enumerate(parts):
                    run = p.add_run(part)
                    if i % 2 != 0: 
                        run.bold = True
            else:
                p.add_run(line)
    
    document.add_page_break() 

if __name__ == "__main__":
    print("Iniciando a modelagem de ameaças STRIDE para arquiteturas de sistema financeiro...")

    # --- DESCRIÇÕES DA ARQUITETURA GERADA PELA VISION ---
    
    # Descrição da AWS
    aws_finance_architecture_description = """
    A arquitetura apresentada descreve uma aplicação web altamente disponível, escalável e segura, implantada na AWS na região de São Paulo (sa-east-1), utilizando uma abordagem Multi-AZ (múltiplas Zonas de Disponibilidade).

    ### 1. Camada de Borda (Edge Services)

    *   **Usuários:** **Usuários SEI** (usuários externos) acessam a aplicação.
    *   **Serviços de Proteção:**
        *   **AWS Shield:** Proteção contra ataques DDoS em nível de rede e transporte.
        *   **Amazon CloudFront:** Atua como CDN e ponto de entrada global, armazenando em cache o conteúdo estático e roteando requisições.
        *   **AWS WAF (Web Application Firewall):** Protege a aplicação contra exploits web comuns e vulnerabilidades, filtrando tráfego malicioso.

    ### 2. Infraestrutura Core (VPC e Multi-AZ)

    *   **Região:** **AWS Cloud / Região sa-east-1 (São Paulo)**.
    *   **Rede Virtual:** **Virtual Private Cloud (VPC)**, fornecendo um ambiente de rede isolado.
    *   **Alta Disponibilidade:** A arquitetura se estende por **Múltiplas Zonas de Disponibilidade (AZ A, B, C)** para resiliência.
    *   **Sub-redes:** Em cada AZ, há:
        *   **Public Subnet:** Contém os Application Load Balancers (ALBs).
        *   **Private Subnet:** Contém os servidores de aplicação e outros recursos internos.

    ### 3. Camada de Aplicação

    *   **Serviços de Rede/Distribuição:**
        *   **Application Load Balancer (ALB):** Presente em cada Public Subnet, distribui o tráfego para as instâncias de aplicação nas Private Subnets.
    *   **Servidores/Processamento:**
        *   **Servidores de Aplicação (SEI/SIP):** Instâncias que rodam a aplicação principal, localizadas nas Private Subnets.
        *   **Auto Scaling (API Server):** Gerencia os servidores SEI/SIP, escalando-os automaticamente com base na demanda.
        *   **Solr:** Uma instância de Solr (para funcionalidade de busca, provavelmente) com seu próprio Auto Scaling.

    ### 4. Camada de Dados e Persistência

    *   **Sistema de Arquivos:**
        *   **Amazon Elastic File System (EFS) Multi-AZ:** Sistema de arquivos compartilhado e escalável (NFS), acessível por múltiplas instâncias em diferentes AZs.
    *   **Bancos de Dados:**
        *   **Amazon RDS (Relational Database Service):**
            *   **Primary:** Banco de dados relacional principal.
            *   **Secondary:** Instância secundária de RDS (réplica de leitura ou standby) para alta disponibilidade.
    *   **Cache:**
        *   **Amazon ElastiCache (memcached) Multi-AZ:** Serviço de cache distribuído para acelerar acesso a dados e reduzir carga no BD.

    ### 5. Serviços de Gerenciamento, Monitoramento e Suporte

    *   **Auditoria:** **AWS CloudTrail:** Registra chamadas de API para auditoria.
    *   **Criptografia:** **AWS Key Management Service (KMS):** Gerencia chaves de criptografia.
    *   **Backup:** **AWS Backup:** Centraliza e automatiza backups.
    *   **Monitoramento:** **Amazon CloudWatch:** Monitora recursos, coleta métricas e logs, cria alarmes e dashboards.
    *   **Comunicação:** **Amazon Simple Email Service (SES):** Serviço de envio de e-mails.

    ---
    """

    # Descrição Azure
    azure_finance_architecture_description = """
    A arquitetura Azure apresentada ilustra uma solução robusta para expor, gerenciar e orquestrar APIs, integrando-se com diversos sistemas de backend. O foco está na segurança, gerenciamento de APIs e orquestração de fluxos de trabalho.

    ### 1. Consumidores e Identidade

    *   **Usuários:** **External User/Application** (Usuário Externo/Aplicação) representa os consumidores das APIs. Eles iniciam requisições HTTP e também interagem com o Developer Portal para criar aplicações e acessar a documentação.
    *   **Gerenciamento de Identidade:** **Microsoft Entra ID (Azure Active Directory)**: Fornece serviços de autenticação para as requisições externas, garantindo que apenas usuários/aplicações autorizados possam acessar as APIs.

    ### 2. Gerenciamento e Exposição de APIs

    *   **Serviços de API:** **Azure API Management (APIM)**: Este é o componente central para o gerenciamento de APIs.
        *   **API Gateway:** Atua como o ponto de entrada único para todas as chamadas de API externas. Ele é responsável por:
            *   Enforçar autenticação (via Entra ID).
            *   Roteamento de requisições.
            *   Aplicação de políticas (cache, rate limiting, transformações).
            *   Criação e publicação de interfaces (definições de API).
        *   **Developer Portal:** Um portal de autoatendimento para desenvolvedores, onde eles podem descobrir APIs, acessar documentação, testar APIs e registrar suas aplicações. As interfaces (definições de API) são publicadas aqui a partir do API Gateway e, opcionalmente, de Logic Apps.

    ### 3. Orquestração e Lógica de Negócios

    *   **Orquestrador/Servidor de Processamento:** **Azure Logic Apps**: Este componente é responsável pela orquestração de workflows e pela execução de lógica de negócios. Após uma requisição API passar pelo API Gateway, ela é roteada para as Logic Apps. Elas podem executar lógica complexa, transformar dados e orquestrar interações com múltiplos sistemas de backend de forma serverless. Interfaces também podem ser publicadas do Logic Apps para o Developer Portal.

    ### 4. Sistemas de Backend

    *   **Serviços Externos:** **Backend Systems**: Estes são os serviços reais que atendem às requisições da API, orquestradas pelas Logic Apps. Podem incluir:
        *   **Azure services:** Outros serviços hospedados no Azure (e.g., Azure Functions, Azure SQL Database, Azure Storage Accounts, etc.).
        *   **SaaS services:** Aplicações de Software como Serviço de terceiros.
        *   **Web services (REST e SOAP):** Serviços web personalizados ou legados que utilizam protocolos RESTful ou SOAP.

    ### 5. Fluxo Geral

    Um usuário ou aplicação externo inicia uma requisição HTTP, que é autenticada pelo Microsoft Entra ID. Após a autenticação, a requisição é direcionada ao API Gateway no Azure API Management. O API Gateway roteia a requisição para o Azure Logic Apps para processamento, que então orquestra os passos necessários, chamando diversos sistemas de backend para atender à requisição. Desenvolvedores utilizam o Developer Portal para descobrir, entender e integrar-se com estas APIs.
    """

    # --- Geração e Salvamento dos Relatórios ---

    # 1. Cria um novo documento Word
    combined_document = Document()
    combined_document.add_heading("Relatório Consolidado de Modelagem de Ameaças STRIDE", level=0)
    combined_document.add_paragraph("Este documento apresenta as análises de modelagem de ameaças STRIDE para múltiplas arquiteturas de sistemas financeiros.")
    combined_document.add_page_break() # Quebra de página inicial para começar a primeira arquitetura

    # 2. Gera e adiciona o relatório AWS ao documento
    print("\n--- Gerando Relatório STRIDE para Arquitetura AWS Financeira ---")
    aws_report = generate_stride_report(aws_finance_architecture_description, "Arquitetura AWS Financeira - Sistema SEI/SIP")
    print(aws_report) # Ainda imprime no console para você ver
    append_markdown_to_docx_document(combined_document, aws_report, "Análise da Arquitetura AWS Financeira - Sistema SEI/SIP")
    print("\n" + "="*80 + "\n")

    # 3. Gera e adiciona o relatório Azure ao documento
    print("\n--- Gerando Relatório STRIDE para Arquitetura Azure Financeira ---")
    azure_report = generate_stride_report(azure_finance_architecture_description, "Arquitetura Azure Financeira - Gerenciamento de APIs")
    print(azure_report) # Ainda imprime no console para você ver
    append_markdown_to_docx_document(combined_document, azure_report, "Análise da Arquitetura Azure Financeira - Gerenciamento de APIs")
    print("\n" + "="*80 + "\n")

    # 4. Salva o documento combinado
    combined_filename = "Relatorio_Consolidado_STRIDE_Financeiro.docx"
    combined_document.save(combined_filename)
    print(f"Relatório consolidado salvo em '{combined_filename}'")

    print("Processo de modelagem de ameaças STRIDE concluído para ambas as arquiteturas.")

Iniciando a modelagem de ameaças STRIDE para arquiteturas de sistema financeiro...

--- Gerando Relatório STRIDE para Arquitetura AWS Financeira ---

Enviando descrição da arquitetura 'Arquitetura AWS Financeira - Sistema SEI/SIP' para a API da OpenAI para análise STRIDE...
# Relatório de Modelagem de Ameaças STRIDE

## Nome da Arquitetura: Arquitetura AWS Financeira - Sistema SEI/SIP

Este relatório identifica ameaças potenciais na arquitetura descrita e sugere mitigações práticas para cada categoria STRIDE.

---

## 1. Spoofing

### Ameaças Potenciais

- **Usuários SEI (usuários externos):** Usuários maliciosos podem tentar se passar por usuários legítimos para acessar informações financeiras sensíveis.

### Impacto

- Acesso não autorizado a dados financeiros críticos, resultando em perda financeira e danos à reputação da organização.

### Mitigações

- **Autenticação Multifator (MFA):** Implementar MFA para todos os usuários, garantindo que um segundo fator seja necessário para aut